# Estudo de caso 6.2 - Design de previsões com dados sobre vendas no Reino Unido

# Configuração

Execute (Run) estas células para instalar os pacotes necessários para a realização do estudo de caso. Tenha paciência, pois isso poderá levar alguns minutos.

<h1 style="color:red;">ATENÇÃO: Podem ocorrer erros ao executar as células abaixo. Mas não se preocupe. É só executar a célula de importação de bibliotecas (duas mais abaixo) e, se aparecer a mensagem "Bibliotecas importadas com sucesso", você pode prosseguir com o estudo de caso.<h1>

In [1]:
!pip uninstall -y folium
#!pip install -q folium==0.2.1
!pip install -q folium==0.9.1
!pip uninstall -y urllib3
!pip install -q urllib3==1.25.4

Found existing installation: folium 0.14.0
Uninstalling folium-0.14.0:
  Successfully uninstalled folium-0.14.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
geemap 0.26.0 requires folium>=0.13.0, but you have folium 0.9.1 which is incompatible.
Found existing installation: urllib3 2.0.4
Uninstalling urllib3-2.0.4:
  Successfully uninstalled urllib3-2.0.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 2.4 MB/s eta 0:00:00


In [2]:
!pip install -q --upgrade pip
!pip uninstall -y featuretools
#!pip install -q featuretools~=0.23.0
!pip install -q featuretools~=0.1.14
!pip uninstall -y pandas
!pip install -q pandas~=1.4.0
print('Bibliotecas instaladas com sucesso!!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.9/140.9 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.4/173.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 45.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2023.6.0 requires fsspec==2023.6.0, but you have fsspec 2023.9.1 which is incompatible.
Found existing installation: pandas 1.5.3
Uninstalling pandas-1.5.3:
  Successfully uninstalled pandas-1.5.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 43.5 MB/s eta 0:00:00
ERROR: pip's dependenc

Esperado um "error" de compatibilidade do google colab 1.0.0 com o pandas 1.4.4 no código anterior - este erro não impacta a execução do caso de uso.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


<h1>Atenção:</h1>

Agora, reinicie o ambiente de execução. Para isso, vá para:

> Ambiente de execução > _Reiniciar ambiente de execução_

na parte superior da tela. Isto irá garantir que suas alterações foram feitas com sucesso.


# Importar

Sincronize sua conta do Google. Para isso, siga o link que aparece na saída da seguinte célula uma vez executada. Copie o código que aparece na tela e insira-o na saída da célula. Assim que visualizar a mensagem: `Google Drive sincronizado com sucesso!`poderá continuar executando o restante das células.

In [4]:
!mkdir uk-retail-data

In [5]:
#!pip uninstall -y pyyaml
#!pip install pyyaml==5.3.1

Esperado um "error" de compatibilidade do flax 0.7.2 - este erro não impacta a execução do caso de uso. Em alguns casos é necessario reiniciar pressionando o botão "RESTART RUNTIME" no código acima, o botão aparecerá no caso de ser necessário.


In [6]:
from google.colab import auth
auth.authenticate_user()

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
data = drive.CreateFile({'id':'1feQJdQqOYUaxeUcC7fj7s-ib3Wuqj4Hf'})
data.GetContentFile('uk-retail-data/customers.csv')
data = drive.CreateFile({'id':'16spEigNWUceuyB5uyaFyo5gW3YX5Fkko'})
data.GetContentFile('uk-retail-data/invoices.csv')
data = drive.CreateFile({'id':'1RlgJaKznx931rnp-vUpki_zHOtkq1eEy'})
data.GetContentFile('uk-retail-data/item_purchases.csv')
data = drive.CreateFile({'id':'1Iwf8zTmLzZyyuhJg3CLc_zW4ZtpoviAy'})
data.GetContentFile('uk-retail-data/items.csv')

data = drive.CreateFile({'id':'1UbV2z7L5vonCz3KFLywGs4U-p2g5gQwy'})
data.GetContentFile('utils.py')

print('Google Drive sincronizado com sucesso!')

Google Drive sincronizado com sucesso!


Importe as bibliotecas necessárias para o desenvolvimento do estudo de caso.

In [7]:
# Ajuste do "Loader" dentro do Featuretools
import os

# Define the path to the original config.py file
config_file_path = '/usr/local/lib/python3.10/dist-packages/featuretools/config.py'

# Read the content of the original file
with open(config_file_path, 'r') as f:
    lines = f.readlines()

# Find the line to replace
for i, line in enumerate(lines):
    if 'config_dict = yaml.load(text)' in line:
        lines[i] = '        config_dict = yaml.load(text, Loader=yaml.FullLoader)\n'

# Write the modified content back to the original file
with open(config_file_path, 'w') as f:
    f.writelines(lines)

In [8]:
import featuretools as ft
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import utils

from utils import (find_training_examples,
                   load_uk_retail_data,
                   engineer_features_uk_retail,
                   preview,
                   feature_importances)

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


In [9]:
# assert ft.__version__ == '0.1.19', 'Certifique-se de ter executado o comando anterior com a versão correta.'
# assert pd.__version__ == '0.20.3', 'Certifique-se de ter executado o comando anterior com a versão correta.'
# print('Versão correta das bibliotecas chave!!')

# Dados

Carregue o banco de dados de vendas de diferentes produtos no Reino Unido. Tenha paciência, pois isso poderá levar alguns minutos.

In [10]:
item_purchases, invoices, items, customers = load_uk_retail_data()
preview(items, 10)

,StockCode,Description,first_item_purchases_time
0,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2010-12-01 08:26:00
1,22752,SET 7 BABUSHKA NESTING BOXES,2010-12-01 08:26:00
2,71053,WHITE METAL LANTERN,2010-12-01 08:26:00
3,84029E,RED WOOLLY HOTTIE WHITE HEART.,2010-12-01 08:26:00
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,2010-12-01 08:26:00
5,84406B,CREAM CUPID HEARTS COAT HANGER,2010-12-01 08:26:00
6,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2010-12-01 08:26:00
7,22632,HAND WARMER RED POLKA DOT,2010-12-01 08:28:00
8,22633,HAND WARMER UNION JACK,2010-12-01 08:28:00
9,21754,HOME BUILDING BLOCK WORD,2010-12-01 08:34:00


In [11]:
entities = {"item_purchases": (item_purchases,
                               "item_purchase_id",
                               "InvoiceDate"),
            "items": (items,
                      "StockCode"),
            "customers": (customers,
                          "CustomerID"),
            "invoices":(invoices,
                        "InvoiceNo",
                        "first_item_purchases_time")
            }
entities

{'item_purchases': (        item_purchase_id InvoiceNo StockCode  Quantity         InvoiceDate  \
  0                      0    536365    85123A         6 2010-12-01 08:26:00   
  1                      1    536365     71053         6 2010-12-01 08:26:00   
  2                      2    536365    84406B         8 2010-12-01 08:26:00   
  3                      3    536365    84029G         6 2010-12-01 08:26:00   
  4                      4    536365    84029E         6 2010-12-01 08:26:00   
  ...                  ...       ...       ...       ...                 ...   
  541904            541904    581587     22613        12 2011-12-09 12:50:00   
  541905            541905    581587     22899         6 2011-12-09 12:50:00   
  541906            541906    581587     23254         4 2011-12-09 12:50:00   
  541907            541907    581587     23255         4 2011-12-09 12:50:00   
  541908            541908    581587     22138         3 2011-12-09 12:50:00   
  
          UnitPrice

In [12]:
relationships = [("customers","CustomerID","invoices","CustomerID"),
                 ("invoices", "InvoiceNo","item_purchases", "InvoiceNo"),
                 ("items", "StockCode","item_purchases", "StockCode")]
relationships

[('customers', 'CustomerID', 'invoices', 'CustomerID'),
 ('invoices', 'InvoiceNo', 'item_purchases', 'InvoiceNo'),
 ('items', 'StockCode', 'item_purchases', 'StockCode')]

## Encontrando exemplos de treinamento

In [13]:
label_times = find_training_examples(item_purchases,
                                     invoices,
                                     prediction_window = pd.Timedelta("14d"),
                                     training_window = pd.Timedelta("21d"),
                                     lead = pd.Timedelta("7d"),
                                     threshold = 5)

/content/utils.py:127: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  label_times = label_times.append(lt)


In [14]:
preview(label_times, 5)

,CustomerID,t_start,cutoff_time,purchases>threshold
0,17505.0,2011-05-18,2011-06-08,False
516,16444.0,2011-05-18,2011-06-08,False
517,16889.0,2011-05-18,2011-06-08,False
518,17613.0,2011-05-18,2011-06-08,True
519,17152.0,2011-05-18,2011-06-08,False


## Geração de variáveis

In [15]:
feature_matrix = engineer_features_uk_retail(entities=entities,
                                             relationships=relationships,
                                             label_times=label_times,
                                             training_window="21d")

/usr/local/lib/python3.10/dist-packages/featuretools/entityset/entity.py:249: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  my_id_vals = pd.Series([]).append(to_append)
/usr/local/lib/python3.10/dist-packages/featuretools/entityset/entity.py:249: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  my_id_vals = pd.Series([]).append(to_append)
/usr/local/lib/python3.10/dist-packages/featuretools/entityset/entity.py:249: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  my_id_vals = pd.Series([]).append(to_append)
/usr/local/lib/python3.10/dist-packages/featuretools/entityset/entity.py:249: FutureWarning: The series.append method is deprecated and will be removed from pandas in a 

In [16]:
preview(feature_matrix, 10)

,MEAN(item_purchases.Quantity),MEAN(item_purchases.UnitPrice),MAX(item_purchases.Quantity),MAX(item_purchases.UnitPrice),STD(item_purchases.Quantity),STD(item_purchases.UnitPrice),MINUTE(first_invoices_time),HOUR(first_invoices_time),DAY(first_invoices_time),WEEK(first_invoices_time),...,MEAN(invoices.STD(item_purchases.Quantity)),MEAN(invoices.STD(item_purchases.UnitPrice)),MAX(invoices.MEAN(item_purchases.Quantity)),MAX(invoices.MEAN(item_purchases.UnitPrice)),MAX(invoices.STD(item_purchases.Quantity)),MAX(invoices.STD(item_purchases.UnitPrice)),STD(invoices.MEAN(item_purchases.Quantity)),STD(invoices.MEAN(item_purchases.UnitPrice)),STD(invoices.MAX(item_purchases.Quantity)),STD(invoices.MAX(item_purchases.UnitPrice))
CustomerID,,,,,,,,,,,,,,,,,,,,,
12353.0,5.000000,6.075000,8,9.95,2.236068,3.911122,47,17,19,20,...,2.236068,3.911122,5.000000,6.075000,2.236068,3.911122,0.000000,0.000000,0.000000,0.000000
12359.0,7.574468,4.914894,24,21.95,5.056046,4.074954,43,12,12,2,...,5.056046,4.074954,7.574468,4.914894,5.056046,4.074954,0.000000,0.000000,0.000000,0.000000
12360.0,9.644444,3.658444,36,40.00,5.453462,6.455818,43,9,23,21,...,5.453462,6.455818,9.644444,3.658444,5.453462,6.455818,0.000000,0.000000,0.000000,0.000000
12380.0,8.138889,2.624444,12,15.00,2.750281,2.491184,49,9,7,23,...,2.750281,2.491184,8.138889,2.624444,2.750281,2.491184,0.000000,0.000000,0.000000,0.000000
12415.0,99.000000,4.765000,216,14.95,70.391761,6.059573,12,11,6,1,...,70.391761,6.059573,99.000000,4.765000,70.391761,6.059573,0.000000,0.000000,0.000000,0.000000
12417.0,5.826087,6.970870,24,16.95,6.505196,5.260192,51,11,17,50,...,6.505196,5.260192,5.826087,6.970870,6.505196,5.260192,0.000000,0.000000,0.000000,0.000000
12423.0,8.074074,3.707407,24,15.00,4.905975,4.286999,54,10,21,51,...,5.561621,4.756742,11.400000,3.864000,7.364781,5.580063,2.040909,0.096091,6.000000,0.000000
12426.0,8.600000,3.936333,25,18.00,5.135497,4.194359,26,12,29,21,...,5.135497,4.194359,8.600000,3.936333,5.135497,4.194359,0.000000,0.000000,0.000000,0.000000
12431.0,10.850000,4.032000,84,8.95,19.246493,2.956003,3,10,1,48,...,7.208697,2.288462,13.588235,4.900000,19.626090,4.050000,8.800536,1.378719,41.515727,3.441253


## Treinamento de um modelo com *Random Forest*

In [17]:
label_times[["CustomerID"]]
X_y = feature_matrix.merge(label_times[["CustomerID",
                                        'purchases>threshold']],
                           right_on="CustomerID",
                           left_index=True)
y = X_y.pop('purchases>threshold')
X_train, X_test, y_train, y_test = train_test_split(feature_matrix,
                                                    y,
                                                    test_size=0.35)

In [18]:
imp = SimpleImputer(missing_values=np.nan,
                    strategy='mean')
imp = imp.fit(X_train)
X_train_imp = imp.transform(X_train)

In [19]:
clf = RandomForestClassifier(random_state=0,
                             n_estimators=100,
                             class_weight="balanced",
                             verbose=True)
clf.fit(X_train_imp, y_train)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.3s


RandomForestClassifier(class_weight='balanced', random_state=0, verbose=True)

## Teste do modelo

In [20]:
X_test_imp = imp.transform(X_test)
predicted_labels = clf.predict(X_test_imp)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s


In [21]:
tn, fp, fn, tp = confusion_matrix(y_test, predicted_labels).ravel()
tn, fp, fn, tp

(215, 5, 52, 3)

In [22]:
feature_importances(clf,feature_matrix.columns, n=15)

1: Feature: WEEK(first_invoices_time), 0.055
2: Feature: STD(invoices.MEAN(item_purchases.UnitPrice)), 0.048
3: Feature: MEAN(invoices.MEAN(item_purchases.UnitPrice)), 0.046
4: Feature: MONTH(first_invoices_time), 0.046
5: Feature: STD(invoices.MAX(item_purchases.UnitPrice)), 0.045
6: Feature: MINUTE(first_invoices_time), 0.045
7: Feature: MEAN(item_purchases.Quantity), 0.044
8: Feature: MAX(invoices.MEAN(item_purchases.Quantity)), 0.043
9: Feature: MEAN(invoices.MEAN(item_purchases.Quantity)), 0.042
10: Feature: STD(item_purchases.UnitPrice), 0.042
11: Feature: MEAN(item_purchases.UnitPrice), 0.041
12: Feature: MEAN(invoices.MAX(item_purchases.UnitPrice)), 0.040
13: Feature: MEAN(invoices.STD(item_purchases.UnitPrice)), 0.038
14: Feature: DAY(first_invoices_time), 0.038
15: Feature: STD(invoices.MEAN(item_purchases.Quantity)), 0.037
